# K=64, two cards, one job each: seed2fp64

pairlatlab60fp64 seed 2 (--keep-best --pair-lab --pair-latent 8 --pair-lab-fp64); pairlab60fp64 seed 2 (--keep-best --pair-lab --pair-lab-fp64), 60 epochs, the K=64 headline command plus each job's flags. T4 with
torch 2.11.0+cu128 pinned and asserted. A run that does not fit the session
stops with a checkpoint for the finish kernel.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
NAME, JOBS, EPOCHS, MAX_SECONDS = 'seed2fp64', [['pairlatlab60fp64', ['--keep-best', '--pair-lab', '--pair-latent', '8', '--pair-lab-fp64'], 2], ['pairlab60fp64', ['--keep-best', '--pair-lab', '--pair-lab-fp64'], 2]], 60, 39000
OUT = "/kaggle/working/results_k64"
CKPT = "/kaggle/working/checkpoints"
LOGS = "/kaggle/working/logs/" + NAME
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
pathlib.Path(LOGS).mkdir(parents=True, exist_ok=True)
for tag, _, _ in JOBS:
    pathlib.Path(OUT, tag).mkdir(parents=True, exist_ok=True)
    pathlib.Path(OUT, tag, "bundle_sha256.txt").write_text('9992ca12c446' + chr(10))
print("data:", DATA, "| bundle", '9992ca12c446')

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"
TODO, DONE = [tuple(j) for j in JOBS], {}

In [ ]:
# Smoke on both cards at once, each card with its own job's flags and seed:
# one epoch on 20k jets. Each must write a result.
import glob, json
smoke = []
for i, (tag, flags, s) in enumerate(TODO[:NGPU]):
    d = "/kaggle/working/smoke/%s/card%d" % (NAME, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE + flags +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", str(s), "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
works = []
for i, ((d, _), rc) in enumerate(zip(smoke, codes)):
    found = glob.glob(d + "/*.json")
    print("card %d: exit %d, result %s" % (i, rc, bool(found)))
    if found:
        r = json.load(open(found[0]))
        print("   %s seed %d: %.0f s, compile %s" % (TODO[i][0], r["seed"], r["walltime_sec"], r.get("compile_check")))
        works.append(i)
    else:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert works or not TODO, "the smoke run failed on every card"
PARALLEL = len(works) >= len(TODO)
print("parallel:" if PARALLEL else "only card(s) %s work: %s alone" % (works, TODO[:1]),
      [j[0] for j in (TODO if PARALLEL else TODO[:1])])

In [ ]:
import glob, time
todo = TODO if PARALLEL else TODO[:1]
procs = []
for i, (tag, flags, s) in enumerate(todo):
    gpu = i if PARALLEL else works[0]
    procs.append((tag, s, launch(["benchmarks.run_top_tagging"] + BASE + flags +
                                 ["--epochs", str(EPOCHS), "--seed", str(s),
                                  "--results-dir", OUT + "/%s/seed%d" % (tag, s),
                                  "--ckpt-dir", CKPT + "/%s/seed%d" % (tag, s),
                                  "--resume", "--max-seconds", str(MAX_SECONDS)],
                                 gpu, LOGS + "/%s_seed%d.log" % (tag, s))))
    print("%s seed %d on card %d" % (tag, s, gpu), flush=True)
BAD = []
for tag, s, p in procs:
    rc = p.wait()
    print("=== %s seed %d: exit %d ===" % (tag, s, rc))
    print(tail(LOGS + "/%s_seed%d.log" % (tag, s), 2000))
    if rc != 0 or not glob.glob(OUT + "/%s/seed%d/*.json" % (tag, s)):
        BAD.append((tag, s))
print("failed:", BAD or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# The K=64 pair-latent reading over three seeds: the committed float32 seeds 0
# and 1 of pairlatlab60 and pairlab60 (separate kernels) and this kernel's
# float64 seed-2 pair. Fixed before the push (K64_CONFIRM_RULE): confirmed if
# the arm's mean gains >= +0.0003 AUC on the reference and every arm run lies
# above every reference run; not confirmed if it gains <= +0.0000; otherwise
# unresolved. The seed-2 pair, started together, differs only by the pair
# latent.
import json, glob, math, statistics
REFS, RULE = {'pairlatlab60': [0.9846450602988644, 0.9849288396121384], 'pairlab60': [0.9841706950893652, 0.9846654727988882]}, (0.0003, 0.0)
POOLED_SD, POOLED_DF = (0.00028969561359768256, 2)   # within-configuration sd of base60 and pairlab60
mean = statistics.mean
recs = {k: v for k, v in DONE.items()}
for tag, _, s in JOBS:
    for f in glob.glob(OUT + "/%s/seed%d/*.json" % (tag, s)):
        recs[(tag, s)] = json.load(open(f))
done = {}
for (tag, s), r in sorted(recs.items()):
    t = r["test_metrics"]; env = r["env"]; h = r["history"]; vl = [e["val_loss"] for e in h]
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    print("%s seed %d: %d epochs (history %d), %.2f h cumulative, %d params | kwargs %s | compile %s | batch order %s"
          % (tag, s, r["epochs_run"], len(h), r["walltime_sec"] / 3600, r["n_params"], r["so3c_kwargs"],
             r.get("compile_check"), r["batch_order"]["epoch1"][:12]))
    print("  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f (lowest %.5f at epoch %d)  val - train %+.5f"
          % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), vl[-1], min(vl),
             h[vl.index(min(vl))]["epoch"], vl[-1] - h[-1]["train_loss"]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): stopped on the session cap" % (r["epochs_run"], EPOCHS))
        continue
    done[tag] = t["test_auc"]

def p_two_sided(t, df):
    try:
        from scipy import stats
        return 2 * stats.t.sf(abs(t), df)
    except Exception:
        return float("nan")

arm_tag, ref_tag = JOBS[0][0], JOBS[1][0]
if arm_tag in done and ref_tag in done:
    a = REFS["pairlatlab60"] + [done[arm_tag]]
    b = REFS["pairlab60"] + [done[ref_tag]]
    d = mean(a) - mean(b)
    tp = d / (POOLED_SD * math.sqrt(1 / len(a) + 1 / len(b)))
    print("seed-2 pair (float64 features, this kernel): pair latent %+.5f" % (done[arm_tag] - done[ref_tag]))
    print("pair latent %s mean %.5f | reference %s mean %.5f | d %+.5f | against the pooled sd %.5f: t %.2f, df %d, p %.3f"
          % (" ".join("%.5f" % x for x in a), mean(a), " ".join("%.5f" % x for x in b), mean(b), d,
             POOLED_SD, tp, POOLED_DF, p_two_sided(tp, POOLED_DF)))
    if d >= RULE[0] and min(a) > max(b):
        verdict = "CONFIRMED at K=64"
    elif d <= RULE[1]:
        verdict = "NOT CONFIRMED"
    else:
        verdict = "unresolved"
    print("verdict (three seeds, arm runs above every reference run: %d of %d): %s"
          % (sum(x > max(b) for x in a), len(a), verdict))
else:
    print("no verdict yet: complete runs", sorted(done))
assert not BAD, "jobs failed: %s" % BAD